# シン・ヒョンジ｜分類ベースライン・JEPX東京エリア価格の急騰予測

チーム・モダピ ・ 分類担当：シン・ヒョンジ ・ 2026-11-16

- **y（正解ラベル）：** `y_spike_rel`：当該枠の価格が前日までの30日間平均の1.5倍以上なら1
- **評価：** 再現率（Recall）、適合率（Precision）、PR-AUC（急騰が少ないため、正解率は主指標としない）
- **比較基準：** ①ランダム予測ベースライン＝急騰率、②ルール＝「前日の同じ時間枠が急騰したら、翌日も急騰と判定」
- **手順：** 0 準備 → 1 急騰データの確認 → 2 モデルA初期版 → 3 ルールベースライン → 4 モデルA改善版 → 5 PR曲線 → 6 ルール＋モデル → 7 モデルB

テスト予測とバックテストは`test_backtest.ipynb`で実施します。

各節に記載した**［11/16の結果］**は、原本に記録されたColabの実行結果です。再実行時に大きく異なる場合は、データやコードの変更を確認してください。

## 0. 準備：Googleドライブの接続とデータの読み込み

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
!pip -q install japanize-matplotlib
import japanize_matplotlib   # グラフの日本語文字化けを防ぐ

from lightgbm import LGBMClassifier
from sklearn.metrics import recall_score, precision_score, average_precision_score, precision_recall_curve

BASE = "/content/drive/MyDrive/kmove 2차 프로젝트/산출물 관리"     # 結果の保存先
df = pd.read_csv(f"{BASE}/D02_전처리&정제 데이터/merged_30min.csv", parse_dates=["date"])
df = df.sort_values(["date", "slot"]).reset_index(drop=True)
print(df.shape)   # (45312, 73)

## 1. 急騰データの確認

**［11/16の結果］**

| split | 30分枠の件数 | 急騰件数 | 急騰率 |
|---|---|---|---|
| train | 18,960 | 712 | 0.038 |
| valid | 8,832 | 539 | 0.061 |
| test | 17,520 | 1,619 | 0.092 |

- 急騰の発生率は学習期間からテスト期間にかけて約2.4倍に増加しています。
- 時間帯別では14時頃から増え、17～18時にピーク（約15%）を迎え、夜間はほぼ0です。
- 月別では7月＞5月＞6月（夏季）で、8月・10～11月はほぼ0です。
- 平日（約5%）は休日（約1.2%）の約4倍です。

In [ ]:
print(df.groupby("split")["y_spike_rel"].agg(["count", "sum", "mean"]).round(3))

In [ ]:
# グラフの共通スタイル（日本語フォントはステップ0で設定）
plt.rcParams.update({"axes.spines.top":False,"axes.spines.right":False,"axes.edgecolor":"#b4b2a9","axes.labelcolor":"#3d3d3a",
 "xtick.color":"#5f5e5a","ytick.color":"#5f5e5a","axes.grid":True,"axes.grid.axis":"y","grid.color":"#ecebe6","grid.linewidth":0.8,"axes.axisbelow":True,
 "axes.titlesize":12,"axes.titleweight":"bold","font.size":10})
BLUE,ORANGE,GRAY,INK="#2a78d6","#eb6834","#888780","#3d3d3a"
tr=df[df.split=="train"]

fig,ax=plt.subplots(2,2,figsize=(13,8))
s=df.groupby("split")["y_spike_rel"].mean().reindex(["train","valid","test"])*100
b=ax[0,0].bar(["学習\n2024.4~2025.4","検証\n2025.5~10","テスト\n2025.11~2026.10"],s.values,color=BLUE,width=0.5)
ax[0,0].bar_label(b,fmt="%.1f%%",color=INK,padding=3); ax[0,0].set_title("期間別の急騰率"); ax[0,0].set_ylabel("急騰率（%）"); ax[0,0].set_ylim(0,11)
h=tr.groupby("hour")["y_spike_rel"].mean()*100
ax[0,1].bar(h.index,h.values,color=BLUE,width=0.7); ax[0,1].set_title("時間帯別の急騰率（学習期間）"); ax[0,1].set_xlabel("時"); ax[0,1].set_ylabel("急騰率（%）"); ax[0,1].set_xticks(range(0,24,2))
i=h.idxmax(); ax[0,1].annotate(f"{i}時 {h[i]:.1f}%",(i,h[i]),xytext=(-62,-4),textcoords="offset points",color=INK)
mo=tr.groupby("month")["y_spike_rel"].mean()*100
ax[1,0].bar(mo.index,mo.values,color=BLUE,width=0.6); ax[1,0].set_title("月別の急騰率（学習期間）"); ax[1,0].set_xlabel("月"); ax[1,0].set_ylabel("急騰率（%）"); ax[1,0].set_xticks(range(1,13))
o=tr.groupby("is_offday")["y_spike_rel"].mean()*100
b=ax[1,1].bar(["平日","休日"],o.values,color=BLUE,width=0.45); ax[1,1].bar_label(b,fmt="%.1f%%",color=INK,padding=3)
ax[1,1].set_title("平日と休日の急騰率（学習期間）"); ax[1,1].set_ylabel("急騰率（%）"); ax[1,1].set_ylim(0,o.max()*1.25)
fig.suptitle("急騰は増加傾向にあり、夕方・夏季・平日に集中",fontsize=14,fontweight="bold",x=0.01,ha="left")
plt.tight_layout(); plt.savefig(f"{BASE}/spike_distribution.png",dpi=150); plt.show()



**急騰時と通常時の平均値［11/16の結果］**

| 特徴量 | 通常時（0） | 急騰時（1） |
|---|---|---|
| temp_c | 17.28 | 22.17 |
| cooling_deg | 1.63 | 4.29 |
| heating_deg | 2.20 | 1.75 |
| price_tokyo_lag1d | 13.30 | 18.26 |
| price_tokyo_mean30d | 13.56 | 12.72 |
| demand_mw_lag1d | 31,494.59 | 37,686.29 |
| solar_mw_lag1d | 2,947.99 | 2,726.59 |

急騰時の`price_tokyo_mean30d`がむしろ低い理由：急騰は「30日間平均の1.5倍以上」と定義されているため、通常価格が低い期間ほど、比較的小さな価格上昇でも急騰と判定されます。

In [ ]:
cols = ["temp_c", "cooling_deg", "heating_deg", "price_tokyo_lag1d",
        "price_tokyo_mean30d", "demand_mw_lag1d", "solar_mw_lag1d"]
print(tr.groupby("y_spike_rel")[cols].mean().round(2).T)

## 2. モデルA初期版（予備率なし・デフォルト設定）

**［11/16の結果］** PR-AUC 0.261（ランダム予測ベースライン0.061）

| しきい値 | 再現率 | 適合率 | アラート件数 |
|---|---|---|---|
| 0.3 | 0.098 | 0.325 | 163 |
| 0.5 | 0.061 | 0.347 | 95 |
| 0.7 | 0.039 | 0.488 | 43 |

問題点：500本の木により学習データへ過剰適合（過学習）し、検証期間では急騰の予測確率がほとんど出なくなりました。ステップ4で改善します。

In [ ]:
# Xから除外する列：正解ラベル、日付・データ分割情報、リークの恐れがある列
drop = ["date", "slot", "slot_start", "split", "y_price", "y_spike_rel",
        "has_occto", "lng_yen_per_t_m1", "regime_2026"]
X_cols = [c for c in df.columns if c not in drop]
colsA  = [c for c in X_cols if not c.startswith("occto_")]   # モデルA：予備率を除外
print("特徴量Xの数：", len(colsA))   # 59

tr = df[df["split"] == "train"]
va = df[df["split"] == "valid"]

m = LGBMClassifier(n_estimators=500, learning_rate=0.05,
                   class_weight="balanced", random_state=42, verbose=-1)
m.fit(tr[colsA], tr["y_spike_rel"])
p = m.predict_proba(va[colsA])[:, 1]
print("PR-AUC:", round(average_precision_score(va["y_spike_rel"], p), 3),
      "(ランダム予測ベースライン:", round(va["y_spike_rel"].mean(), 3), ")")
for t in [0.3, 0.5, 0.7]:
    pred = (p >= t).astype(int)
    print(f"しきい値 {t}: 再現率 {recall_score(va['y_spike_rel'], pred):.3f}, "
          f"適合率 {precision_score(va['y_spike_rel'], pred, zero_division=0):.3f}, アラート {pred.sum()}")

## 3. ルールによるベースライン：「前日の同じ時間枠が急騰したら、翌日も急騰と判定」

**［11/16の結果］** 再現率0.490、適合率0.508、アラート520件。初期版モデルより強力なベースラインとなっています。

In [ ]:
base = (va["price_tokyo_lag1d"] >= 1.5 * va["price_tokyo_mean30d"]).astype(int)
print(f"ルール ベースライン: 再現率 {recall_score(va['y_spike_rel'], base):.3f}, "
      f"適合率 {precision_score(va['y_spike_rel'], base, zero_division=0):.3f}, アラート {base.sum()}")

## 4. モデルAの改善：比率特徴量4つ＋正則化

- **比率特徴量：** 急騰の定義は「価格÷30日間平均≧1.5」です。木モデルが2列の比率を自動生成するわけではないため、比率を特徴量として追加します（すべて前日までの値でありリークなし）。
- **正則化：** 木の数↓、葉の数↓、葉当たりの最小データ数↑として過学習を抑えます。

**［11/16の結果］** PR-AUC 0.261 → **0.402**

| しきい値 | 再現率 | 適合率 | アラート件数 |
|---|---|---|---|
| 0.3 | 0.677 | 0.313 | 1,165 |
| 0.5 | 0.553 | 0.391 | 762 |
| 0.6 | 0.473 | 0.422 | 604 |
| 0.7 | 0.380 | 0.464 | 442 |
| 0.8 | 0.280 | 0.557 | 271 |

In [ ]:
df["ratio_lag1d"]  = df["price_tokyo_lag1d"]  / df["price_tokyo_mean30d"]
df["ratio_lag7d"]  = df["price_tokyo_lag7d"]  / df["price_tokyo_mean30d"]
df["ratio_mean7d"] = df["price_tokyo_mean7d"] / df["price_tokyo_mean30d"]
df["spike_lag1d"]  = (df["ratio_lag1d"] >= 1.5).astype(int)   # 前日の同じ時間枠で急騰したか

X_cols = [c for c in df.columns if c not in drop]
colsA  = [c for c in X_cols if not c.startswith("occto_")]
tr = df[df["split"] == "train"]; va = df[df["split"] == "valid"]

params = dict(n_estimators=300, learning_rate=0.03, num_leaves=15, min_child_samples=100,
              reg_lambda=1.0, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
              class_weight="balanced", importance_type="gain", random_state=42, verbose=-1)
m = LGBMClassifier(**params)
m.fit(tr[colsA], tr["y_spike_rel"])

p = m.predict_proba(va[colsA])[:, 1]
print("PR-AUC:", round(average_precision_score(va["y_spike_rel"], p), 3))
for t in [0.3, 0.5, 0.6, 0.7, 0.8]:
    pred = (p >= t).astype(int)
    print(f"しきい値 {t}: 再現率 {recall_score(va['y_spike_rel'], pred):.3f}, "
          f"適合率 {precision_score(va['y_spike_rel'], pred, zero_division=0):.3f}, アラート {pred.sum()}")

In [ ]:
imp = pd.Series(m.feature_importances_, index=colsA).sort_values()
imp.tail(15).plot.barh(figsize=(7, 6), color="#E24B4A")
plt.title("急騰分類・重要特徴量TOP15（モデルA、gain基準）")
plt.tight_layout()
plt.savefig(f"{BASE}/feature_importance.png", dpi=150)
plt.show()

## 5. PR曲線：モデルAとルールの比較

**［11/16の結果］** ルール（赤い点）は再現率0.49付近でモデルの曲線より上にあり、同じ再現率ではルールの方が適合率に優れます。一方、モデルはしきい値を調整することで再現率を約0.68まで引き上げられます。

In [ ]:
prec, rec, _ = precision_recall_curve(va["y_spike_rel"], p)
base = (va["ratio_lag1d"] >= 1.5).astype(int)

plt.figure(figsize=(6, 5))
plt.plot(rec, prec, color="#185FA5", label=f"分類モデルA (PR-AUC {average_precision_score(va['y_spike_rel'], p):.3f})")
plt.scatter(recall_score(va["y_spike_rel"], base), precision_score(va["y_spike_rel"], base),
            color="#E24B4A", s=80, zorder=3, label="ルール：前日の急騰 → 翌日の急騰")
plt.axhline(va["y_spike_rel"].mean(), color="gray", ls="--", label=f"ランダム予測ベースライン ({va['y_spike_rel'].mean():.3f})")
plt.xlabel("再現率（急騰を見逃さない割合）"); plt.ylabel("適合率（アラートが正しい割合）")
plt.title("急騰分類・検証期間のPR曲線"); plt.legend(); plt.tight_layout()
plt.savefig(f"{BASE}/pr_curve.png", dpi=150)
plt.show()

## 6. ルールとモデルAを組み合わせる

「ルールが急騰と判定するか、モデルの予測確率がしきい値以上ならアラートを出す」

**［11/16の結果］**

| 手法 | 再現率 | 適合率 | アラート件数 |
|---|---|---|---|
| ルールのみ | 0.490 | 0.508 | 520 |
| ルールORモデル≧0.6 | 0.627 | 0.411 | 822 |
| ルールORモデル≧0.7 | 0.581 | 0.441 | 709 |
| ルールORモデル≧0.8 | 0.549 | 0.482 | 614 |
| ルールORモデル≧0.9 | 0.510 | 0.507 | 542 |

→ 0.8はバランスの取れたしきい値です。急騰539件のうち、ルールのみでは264件、0.8での組み合わせでは296件を検知（＋32件）。

In [ ]:
rule = (va["ratio_lag1d"] >= 1.5).values
y = va["y_spike_rel"]
print(f"ルールのみ: 再現率 {recall_score(y, rule):.3f}, 適合率 {precision_score(y, rule):.3f}, アラート {rule.sum()}")
for t in [0.6, 0.7, 0.8, 0.9]:
    hyb = rule | (p >= t)
    print(f"ルール OR モデル≥{t}: 再現率 {recall_score(y, hyb):.3f}, "
          f"適合率 {precision_score(y, hyb):.3f}, アラート {hyb.sum()}")

## 7. モデルB（予備率を追加）

予備率のデータは2025年4月から利用可能なため、**学習：2025.4～8／検証：2025.9～10**に分割します。モデル比較には同じ検証期間を使用します。

**［11/16の結果］** 学習7,344枠（急騰433件）／検証2,928枠（急騰152件）、ランダム予測ベースライン0.052

- PR-AUC：モデルA 0.375 → **モデルB 0.550**（予備率の追加効果）

| 手法 | 再現率 | 適合率 | アラート件数 | 検知した急騰件数（全152件） |
|---|---|---|---|---|
| ルールのみ | 0.539 | 0.543 | 151 | 82 |
| ルールORモデルA≧0.7 | 0.586 | 0.527 | 169 | 89 |
| ルールORモデルA≧0.8 | 0.579 | 0.543 | 162 | 88 |
| ルールORモデルA≧0.9 | 0.539 | 0.543 | 151 | 82 |
| ルールORモデルB≧0.7 | 0.678 | 0.435 | 237 | 103 |
| ルールORモデルB≧0.8 | 0.658 | 0.483 | 207 | 100 |
| ルールORモデルB≧0.9 | 0.599 | 0.511 | 178 | 91 |

注意：検証期間は2か月間（急騰152件）しかないため、指標が変動しやすい点に注意してください。また、モデルBは主に夏季のデータで学習しています。

In [ ]:
dB  = df[df["has_occto"] == 1]
trB = dB[dB["date"] < "2025-09-01"]
vaB = dB[(dB["date"] >= "2025-09-01") & (dB["split"] == "valid")]
print("学習", len(trB), "枠 (急騰", trB["y_spike_rel"].sum(), ") / 検証", len(vaB), "枠 (急騰", vaB["y_spike_rel"].sum(), ")")

mB = LGBMClassifier(**params)
mB.fit(trB[X_cols], trB["y_spike_rel"])   # X_cols＝予備率（occto_）を含む全特徴量

yB    = vaB["y_spike_rel"]
pB    = mB.predict_proba(vaB[X_cols])[:, 1]
pA_B  = m.predict_proba(vaB[colsA])[:, 1]          # モデルAを同一期間に適用
ruleB = (vaB["ratio_lag1d"] >= 1.5).values

print(f"\n[同一期間2025.9～10の比較]  ランダム予測ベースライン {yB.mean():.3f}")
print(f"PR-AUC  モデルA {average_precision_score(yB, pA_B):.3f}  vs  モデルB {average_precision_score(yB, pB):.3f}")
print(f"ルールのみ: 再現率 {recall_score(yB, ruleB):.3f}, 適合率 {precision_score(yB, ruleB):.3f}, アラート {ruleB.sum()}")
for name, prob in [("A", pA_B), ("B", pB)]:
    for t in [0.7, 0.8, 0.9]:
        hyb = ruleB | (prob >= t)
        print(f"ルール OR モデル{name}≥{t}: 再現率 {recall_score(yB, hyb):.3f}, "
              f"適合率 {precision_score(yB, hyb):.3f}, アラート {hyb.sum()}")

## 8. 結果の可視化（発表用）

ステップ2～7で得られた**［11/16の結果］**の数値を使ったグラフ3点です。再実行によって結果が変わった場合は、以下のハードコードされた値も更新してください。

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4.5),sharey=True)
for a,(title,names,vals) in zip(ax,[("検証2025.5～10・モデルAの改善過程",["ランダム予測ベースライン","モデルA初期版","モデルA改善版\n（比率特徴量＋正則化）"],[0.061,0.261,0.402]),
                                    ("検証2025.9～10・予備率の効果",["ランダム予測ベースライン","モデルA\n（予備率なし）","モデルB\n（予備率あり）"],[0.052,0.375,0.550])]):
    b=a.bar(names,vals,color=[GRAY,BLUE,BLUE],width=0.5); a.bar_label(b,fmt="%.3f",color=INK,padding=3); a.set_title(title,loc="left")
ax[0].set_ylabel("PR-AUC（高いほど良い）"); ax[0].set_ylim(0,0.65)
fig.suptitle("PR-AUC：改善後0.40、予備率を追加すると0.55（ランダム予測の10.6倍）",fontsize=14,fontweight="bold",x=0.01,ha="left")
plt.tight_layout(); plt.savefig(f"{BASE}/pr_auc_comparison.png",dpi=150); plt.show()

plt.rcParams["axes.grid.axis"]="both"
fig,ax=plt.subplots(1,2,figsize=(13,5.2),sharey=True)
def line(a,pts,c,lab,skip=(),dy=6):
    a.plot([r for _,r,_ in pts],[p for *_,p in pts],"-o",color=c,lw=2,ms=8,mec="white",mew=2,label=lab)
    for t,r,p in pts:
        if t not in skip: a.annotate(f"{t}",(r,p),xytext=(6,dy),textcoords="offset points",color=INK,fontsize=9)
a=ax[0]
line(a,[(0.6,.627,.411),(0.7,.581,.441),(0.8,.549,.482),(0.9,.510,.507)],BLUE,"ルール OR モデルA")
a.plot(.490,.508,"s",color=INK,ms=10,mec="white",mew=2,label="ルールのみ（前日の急騰 → 翌日の急騰）")
a.set_title("検証2025.5～10（急騰539件）",loc="left"); a.set_xlabel("再現率（急騰を見逃さない割合）"); a.set_ylabel("適合率（アラートが正しい割合）"); a.legend(frameon=False,loc="lower left")
a=ax[1]
line(a,[(0.7,.586,.527),(0.8,.579,.543),(0.9,.539,.543)],BLUE,"ルール OR モデルA",skip=(0.9,),dy=-14)
line(a,[(0.7,.678,.435),(0.8,.658,.483),(0.9,.599,.511)],ORANGE,"ルール OR モデルB")
a.plot(.539,.543,"s",color=INK,ms=10,mec="white",mew=2,label="ルールのみ（モデルAの0.9と同じ点）")
a.set_title("検証2025.9～10（急騰152件）",loc="left"); a.set_xlabel("再現率（急騰を見逃さない割合）"); a.legend(frameon=False,loc="lower left")
for a_ in ax: a_.set_xlim(0.45,0.72); a_.set_ylim(0.38,0.58)
fig.suptitle("ルールにモデルを加えると適合率を大幅に下げずに急騰をより多く検知（点の数字＝しきい値）",fontsize=13,fontweight="bold",x=0.01,ha="left")
plt.tight_layout(); plt.savefig(f"{BASE}/rule_vs_model.png",dpi=150); plt.show()

plt.rcParams["axes.grid.axis"]="y"
fig,a=plt.subplots(figsize=(8,4.5))
names=["ルールのみ","ルール＋モデルA\n（しきい値0.8）","ルール＋モデルB\n（しきい値0.8）"]; caught=[82,88,100]; alarms=[151,162,207]
b=a.bar(names,caught,color=[GRAY,BLUE,ORANGE],width=0.5)
a.bar_label(b,labels=[f"{c}件 / アラート {n}件" for c,n in zip(caught,alarms)],color=INK,padding=3)
a.axhline(152,color="#b4b2a9",ls="--",lw=1); a.text(2.3,154,"急騰総数152件",ha="right",color="#5f5e5a",fontsize=9)
a.set_ylim(0,170); a.set_ylabel("検知した急騰件数")
a.set_title("検証2025.9～10：ルール＋モデルBで急騰検知が22%増（82→100件）",loc="left")
plt.tight_layout(); plt.savefig(f"{BASE}/spike_detection_counts.png",dpi=150); plt.show()
